# Multi-Class Text Classification with RNNs — Department Prediction

**Goal**: Predict the **Department Name** (Tops, Dresses, Bottoms, Intimate, Jackets, Trend) from review text.

### Notebook Outline
1. Data Loading & Class Distribution Analysis
2. Text Preprocessing & Vocabulary
3. PyTorch Dataset & DataLoader
4. Model Architectures (BiLSTM + MaxPool, BiLSTM + Attention, BiGRU + AvgPool)
5. Training with Class-Weighted Loss
6. Evaluation & Per-Class Analysis
7. Attention Visualization

In [ ]:
import os
import re
import string
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

---
## 1. Data Loading & Class Distribution

In [ ]:
DATA_PATH = os.path.join('Dataset', 'Womens Clothing E-Commerce Reviews.csv')
df = pd.read_csv(DATA_PATH, index_col=0)
df = df.dropna(subset=['Review Text', 'Department Name'])
print(f"Dataset shape: {df.shape}")
print(f"\nDepartment distribution:")
print(df['Department Name'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
dept_counts = df['Department Name'].value_counts()
colors = sns.color_palette('husl', len(dept_counts))
dept_counts.plot.bar(ax=axes[0], color=colors)
axes[0].set_title('Department Distribution', fontsize=13)
axes[0].set_xlabel('Department')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Average review length by department
df['word_count'] = df['Review Text'].str.split().str.len()
avg_len = df.groupby('Department Name')['word_count'].mean().sort_values(ascending=False)
avg_len.plot.bar(ax=axes[1], color=colors)
axes[1].set_title('Average Review Length by Department', fontsize=13)
axes[1].set_xlabel('Department')
axes[1].set_ylabel('Words')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Average rating by department
fig, ax = plt.subplots(figsize=(8, 4))
dept_rating = df.groupby('Department Name')['Rating'].mean().sort_values()
dept_rating.plot.barh(ax=ax, color=sns.color_palette('coolwarm', len(dept_rating)))
ax.set_title('Average Rating by Department')
ax.set_xlabel('Mean Rating')
ax.axvline(df['Rating'].mean(), color='black', linestyle='--', label=f'Overall: {df["Rating"].mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Text Preprocessing & Vocabulary

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['clean_text'] = df['Review Text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 0]

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['Department Name'])
num_classes = len(le.classes_)
class_names = le.classes_.tolist()

print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")
print(f"Cleaned dataset: {df.shape[0]:,} reviews")

In [ ]:
# Split data
texts = df['clean_text'].tolist()
labels = df['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# Build vocabulary
class Vocabulary:
    PAD_TOKEN = '<PAD>'
    UNK_TOKEN = '<UNK>'

    def __init__(self, max_size=25_000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.token2idx = {self.PAD_TOKEN: 0, self.UNK_TOKEN: 1}
        self.idx2token = {0: self.PAD_TOKEN, 1: self.UNK_TOKEN}

    def build(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(t.split())
        idx = len(self.token2idx)
        for word, freq in counter.most_common(self.max_size):
            if freq < self.min_freq:
                continue
            if word not in self.token2idx:
                self.token2idx[word] = idx
                self.idx2token[idx] = word
                idx += 1

    def encode(self, text, max_len):
        tokens = text.split()[:max_len]
        indices = [self.token2idx.get(t, 1) for t in tokens]
        return indices + [0] * (max_len - len(indices))

    def __len__(self):
        return len(self.token2idx)

vocab = Vocabulary(max_size=25_000, min_freq=2)
vocab.build(X_train)
print(f"Vocabulary size: {len(vocab):,}")

---
## 3. PyTorch Dataset & DataLoader

In [ ]:
MAX_LEN = 200
BATCH_SIZE = 64

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.vocab.encode(self.texts[idx], self.max_len)
        return (
            torch.tensor(enc, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )

train_ds = ReviewDataset(X_train, y_train, vocab)
val_ds   = ReviewDataset(X_val, y_val, vocab)
test_ds  = ReviewDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# Compute class weights for imbalanced dataset
class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * num_classes
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print(f"Class counts (train): {class_counts}")
print(f"Class weights: {class_weights.round(3)}")

---
## 4. Model Architectures

We compare three approaches:
1. **BiLSTM + Max Pooling** — Takes the max activation across all timesteps
2. **BiLSTM + Self-Attention** — Learns which timesteps matter most
3. **BiGRU + Average Pooling** — Simple mean of all hidden states

In [ ]:
EMBED_DIM  = 128
HIDDEN_DIM = 128
N_LAYERS   = 2
DROPOUT    = 0.3


class BiLSTMMaxPool(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 n_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, _ = self.lstm(embedded)
        pooled, _ = output.max(dim=1)
        return self.classifier(pooled)


class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 n_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attention=False):
        mask = (x != 0).unsqueeze(-1).float()
        embedded = self.dropout(self.embedding(x))
        output, _ = self.lstm(embedded)

        attn_scores = self.attention(output)
        attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_weights = F.softmax(attn_scores, dim=1)
        context = (output * attn_weights).sum(dim=1)

        logits = self.classifier(context)
        if return_attention:
            return logits, attn_weights.squeeze(-1)
        return logits


class BiGRUAvgPool(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 n_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x != 0).unsqueeze(-1).float()
        embedded = self.dropout(self.embedding(x))
        output, _ = self.gru(embedded)
        lengths = mask.sum(dim=1).clamp(min=1)
        avg_pool = (output * mask).sum(dim=1) / lengths
        return self.classifier(avg_pool)


# Print param counts
for name, cls in [('BiLSTM+MaxPool', BiLSTMMaxPool),
                   ('BiLSTM+Attention', BiLSTMAttention),
                   ('BiGRU+AvgPool', BiGRUAvgPool)]:
    m = cls(len(vocab), EMBED_DIM, HIDDEN_DIM, num_classes, N_LAYERS, DROPOUT)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:20s}: {n:>10,} parameters")

---
## 5. Training with Class-Weighted Loss

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in loader:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(texts)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for texts, labels in loader:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        logits = model(texts)
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

In [ ]:
EPOCHS = 5
LR = 1e-3

models_config = {
    'BiLSTM+MaxPool': BiLSTMMaxPool,
    'BiLSTM+Attention': BiLSTMAttention,
    'BiGRU+AvgPool': BiGRUAvgPool,
}

history = {}
best_models = {}

for name, ModelClass in models_config.items():
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"{'='*60}")

    model = ModelClass(
        vocab_size=len(vocab), embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM, num_classes=num_classes,
        n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    train_losses, val_losses, val_accs = [], [], []
    best_val_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_models[name] = model.state_dict().copy()

        print(f"  Epoch {epoch}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"{elapsed:.1f}s")

    history[name] = {'train_loss': train_losses, 'val_loss': val_losses, 'val_acc': val_accs}

---
## 6. Evaluation & Per-Class Analysis

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, h in history.items():
    axes[0].plot(h['train_loss'], label=name, linewidth=2)
    axes[1].plot(h['val_loss'], label=name, linewidth=2)
    axes[2].plot(h['val_acc'], label=name, linewidth=2)

for ax, title in zip(axes, ['Train Loss', 'Val Loss', 'Val Accuracy']):
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Test evaluation
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
test_results = {}

for name, ModelClass in models_config.items():
    model = ModelClass(
        vocab_size=len(vocab), embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM, num_classes=num_classes,
        n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)
    model.load_state_dict(best_models[name])

    test_loss, test_acc, preds, labels_arr = evaluate(model, test_loader, criterion)
    test_results[name] = {'acc': test_acc, 'preds': preds, 'labels': labels_arr}

    print(f"\n{'='*50}")
    print(f"{name}  —  Test Accuracy: {test_acc:.4f}")
    print(f"{'='*50}")
    print(classification_report(labels_arr, preds, target_names=class_names))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, res) in zip(axes, test_results.items()):
    cm = confusion_matrix(res['labels'], res['preds'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_pct, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(f'{name}\nAcc={res["acc"]:.3f}', fontsize=12)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy comparison
per_class = {}
for name, res in test_results.items():
    correct_per_class = []
    for c in range(num_classes):
        mask = res['labels'] == c
        acc = (res['preds'][mask] == c).mean() if mask.sum() > 0 else 0
        correct_per_class.append(acc)
    per_class[name] = correct_per_class

per_class_df = pd.DataFrame(per_class, index=class_names)

per_class_df.plot.bar(figsize=(10, 5), width=0.8)
plt.title('Per-Class Accuracy Comparison', fontsize=13)
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Attention Visualization

The BiLSTM+Attention model can reveal **which words** the model focuses on for each prediction.

In [ ]:
def visualize_attention(text, model, vocab, le, max_len=MAX_LEN, top_k=40):
    """Show attention weights over words for a given review."""
    model.eval()
    cleaned = clean_text(text)
    words = cleaned.split()[:max_len]
    encoded = vocab.encode(cleaned, max_len)
    tensor = torch.tensor([encoded], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits, attn_weights = model(tensor, return_attention=True)

    pred_class = le.classes_[logits.argmax(1).item()]
    attn = attn_weights[0, :len(words)].cpu().numpy()

    # Only show top_k words
    display_words = words[:top_k]
    display_attn = attn[:top_k]

    fig, ax = plt.subplots(figsize=(14, 3))
    colors = plt.cm.Reds(display_attn / display_attn.max())
    ax.bar(range(len(display_words)), display_attn, color=colors)
    ax.set_xticks(range(len(display_words)))
    ax.set_xticklabels(display_words, rotation=65, ha='right', fontsize=8)
    ax.set_ylabel('Attention Weight')
    ax.set_title(f'Predicted Department: {pred_class}', fontsize=13)
    plt.tight_layout()
    plt.show()

# Load attention model
attn_model = BiLSTMAttention(
    vocab_size=len(vocab), embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM, num_classes=num_classes,
    n_layers=N_LAYERS, dropout=DROPOUT,
).to(DEVICE)
attn_model.load_state_dict(best_models['BiLSTM+Attention'])

# Pick sample reviews from different departments
for dept in ['Dresses', 'Tops', 'Bottoms']:
    sample = df[df['Department Name'] == dept]['Review Text'].iloc[0]
    print(f"\nDepartment: {dept}")
    print(f"Review: {sample[:120]}...")
    visualize_attention(sample, attn_model, vocab, le)

---
## Summary

### Architecture Comparison

| Model | Pooling Strategy | Key Insight |
|-------|-----------------|-------------|
| BiLSTM + MaxPool | Takes strongest signal per feature | Good at detecting key phrases |
| BiLSTM + Attention | Learns importance per timestep | Interpretable; focuses on informative words |
| BiGRU + AvgPool | Averages all timesteps | Smooth representation; fewer parameters |

### Key Takeaways
- Class-weighted loss helps with the imbalanced "Trend" class (only ~119 samples)
- Attention mechanism provides **interpretability** — we can see which words drive predictions
- Department-specific vocabulary (e.g., "dress", "pants", "bra") is the strongest signal
- All three models struggle most with **Trend** due to very few training samples